# Aula 1, Como LLMs funcionam

Notebook executável que acompanha a aula [01-como-llms-funcionam.md](../../lessons/modulo-07-llms/01-como-llms-funcionam.md).

## O que vamos fazer aqui

Um LLM gera texto prevendo a próxima palavra e amostrando dessa distribuição. Vamos
demonstrar a amostragem por temperatura sobre uma pequena lista de logits, vendo a
distribuição mudar de quase determinística a bem variada. Depois, um caminho opcional gera
texto com um modelo de verdade via Ollama.

## Amostragem por temperatura

A temperatura divide os logits antes da softmax. Valores baixos concentram a probabilidade
no token mais provável; valores altos achatam a distribuição.

In [ ]:
import numpy as np


def softmax_temperatura(logits, T):
    """Converte logits em probabilidades, com temperatura T."""
    z = np.array(logits) / T
    z -= z.max()
    e = np.exp(z)
    return e / e.sum()


logits = [2.0, 1.0, 0.5, 0.1]   # quatro candidatos a próxima palavra

print("Distribuição da próxima palavra conforme a temperatura:")
for T in [0.2, 1.0, 2.0]:
    p = softmax_temperatura(logits, T)
    print(f"  T = {T}:  {np.round(p, 3)}   prob do mais provável = {p.max():.3f}")

Com T=0,2, o token mais provável fica com cerca de 0,99 (geração quase
determinística). Com T=1,0, a distribuição é a do modelo (topo ~0,57). Com T=2,0, ela se
achata (topo ~0,41), abrindo espaço para escolhas variadas. Esse único parâmetro separa uma
resposta sóbria de uma criativa.

## Caminho opcional, gerar com um LLM de verdade

Esta célula gera a mesma resposta em duas temperaturas via Ollama, para sentir o efeito na
prática. Roda apenas se o Ollama estiver disponível.

In [ ]:
try:
    import ollama

    pergunta = "Em uma frase, dê uma dica de estudo para um aluno."
    for T in [0.2, 1.2]:
        r = ollama.chat(
            model="llama3.1",
            messages=[{"role": "user", "content": pergunta}],
            options={"temperature": T},
        )
        print(f"[temperatura {T}] {r['message']['content'].strip()}\n")
except Exception as erro:
    print("Ollama não disponível agora. Veja docs/setup.md. Detalhe:", erro)

Com temperatura baixa, a resposta tende a ser mais previsível e segura; com
temperatura alta, mais variada e criativa. Para o projeto, gere respostas em três
temperaturas e diga qual faixa usaria em um assistente educacional.